## Calcular el valor presente de las cuotas pendientes a mitad del préstamo con el método seleccionado. 

In [6]:
import pandas as pd
import numpy as np
import os

In [8]:
df = pd.read_excel(r"..\Datos\Limpios\información_préstamos_limpio.xlsx")

In [9]:
#Valor presente de las cuotas pendientes
#Sistema francés – préstamo representativo (mediana)
# Selección del préstamo representativo (mediana)
df['Interes_Anual'] = df['Ratio_Interes'] /100

prestamo = df.loc[
    (df["Monto_Inicial"] - df["Monto_Inicial"].median()).abs().idxmin()
]

capital = prestamo["Monto_Inicial"]
interes_anual = prestamo["Interes_Anual"]
n_periodos = int(prestamo["Duracion"])

print("Préstamo seleccionado: representativo (mediana):")
print(f"Capital inicial: {capital:,.2f} €")
print(f"Interés anual: {interes_anual * 100:.2f} %")
print(f"Duración: {n_periodos} meses")

Préstamo seleccionado: representativo (mediana):
Capital inicial: 40,000.00 €
Interés anual: 8.59 %
Duración: 60 meses


In [10]:
def amortizacion_frances(capital, interes_anual, n_periodos):    
    i = interes_anual / 12
    cuota = capital * (i * (1 + i)**n_periodos) / ((1 + i)**n_periodos - 1)

    saldo = capital
    tabla = []

    for t in range(1, n_periodos + 1):
        intereses = saldo * i
        amortizacion = cuota - intereses
        saldo -= amortizacion

        tabla.append([t, cuota, intereses, amortizacion, saldo])

    return pd.DataFrame(
        tabla,
        columns=["Periodo", "Cuota", "Intereses", "Amortizacion", "Saldo_pendiente"]
    )

In [11]:
#Crear tabla de amortización
frances = amortizacion_frances(capital, interes_anual, n_periodos)

In [12]:
#Tipo de interés por periodo (mensual)
i = interes_anual / 12

#Cálculo de la cuota constante (sistema francés)
cuota = capital * (i * (1 + i)**n_periodos) / ((1 + i)**n_periodos - 1)

#Punto intermedio del préstamo
mitad = n_periodos // 2
periodos_pendientes = n_periodos - mitad

# Valor presente de las cuotas pendientes
vp_cuotas_pendientes = cuota * (1 - (1 + i)**(-periodos_pendientes)) / i

print("\nResultados")
print(f"Cuota mensual (sistema francés): {cuota:,.2f} €")
print(f"Periodo de cálculo: {mitad}")
print(f"Cuotas pendientes: {periodos_pendientes}")
print(f"Valor presente de las cuotas pendientes: {vp_cuotas_pendientes:,.2f} €")



Resultados
Cuota mensual (sistema francés): 822.40 €
Periodo de cálculo: 30
Cuotas pendientes: 30
Valor presente de las cuotas pendientes: 22,131.72 €


In [13]:
#Validación con tabla de amortización 
saldo_pendiente_mitad = frances.loc[
    frances["Periodo"] == mitad, "Saldo_pendiente"
].values[0]

print("\nVALIDACIÓN ALTERNATIVA (OBJETIVO 2)")
print(f"Saldo pendiente en el periodo medio: {saldo_pendiente_mitad:,.2f} €")
print(f"Valor presente de las cuotas pendientes: {vp_cuotas_pendientes:,.2f} €")
print(
    f"Diferencia absoluta: {abs(saldo_pendiente_mitad - vp_cuotas_pendientes):,.4f} €"
)

#Conclusión
if abs(saldo_pendiente_mitad - vp_cuotas_pendientes) < 1e-2:
    print("La validación es correcta: el valor presente coincide con el saldo pendiente.")
else:
    print("Existe una diferencia: revisar fórmulas o tabla de amortización.")



VALIDACIÓN ALTERNATIVA (OBJETIVO 2)
Saldo pendiente en el periodo medio: 22,131.72 €
Valor presente de las cuotas pendientes: 22,131.72 €
Diferencia absoluta: 0.0000 €
La validación es correcta: el valor presente coincide con el saldo pendiente.
